In [2]:
import requests
from datetime import datetime, timezone
import urllib3
from zoneinfo import ZoneInfo
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import pandas as pd

pd.set_option("display.max_colwidth", None)

def parseIbkrTime(timeString: str) -> datetime:
    # Used for metadata returned
    return datetime.strptime(timeString, "%Y%m%d-%H:%M:%S").replace(tzinfo=timezone.utc)

def formatTime(time):
    if type(time) is str:
        time = parseIbkrTime(time)

    if type(time) is int:
        time = time / 1000.0
        time = datetime.fromtimestamp(time, tz=timezone.utc)

    if type(time) is datetime:
        if time.tzinfo is None:
            time = time.replace(tzinfo=timezone.utc)
        ny_time = time.astimezone(ZoneInfo("America/New_York"))
        fmt = "%Y-%m-%d %a %I:%M:%S %p"
        time = f"(utc) {time.strftime(fmt)}, (NY) {ny_time.strftime(fmt)}"
        
    return time

# The input date time is in utc and api expects utc I think
def dataQuery(
    conid: int,
    bar: str,
    startTime: datetime,
    period: str = None,
    outsideRth: bool = False
):
    startTime = startTime.strftime('%Y%m%d-%H:%M:%S')

    params = {
        'conid' : conid,
        'bar' : bar,
        'period' : period,
        'startTime' : startTime,
        'outsideRth' : outsideRth
    }
    r = requests.get("https://localhost:5000/v1/api/iserver/marketdata/history", params=params, verify=False)

    data = r.json()
    #print(data)
    formattedTime = formatTime(data['startTime'])

    print(f"{'startTime:':<20} {formattedTime}")
    print(f"{'chartPanStartTime:':<20} {formatTime(data['chartPanStartTime'])}")
    print(f"{'timePeriod:':<20} {data['timePeriod']}")
    print(f"{'barLength:':<20} {data['barLength']}")
    df = pd.DataFrame(data['data'])
    df['t'] = df['t'].apply(formatTime)
    return df


Bars smaller than 1 min are allowed and work. However, if you give a period of less than 1 min (60 seconds) then those 15s bars are placed one minute apart. Behaviour is inconsistent

In [34]:
# Data is returned for our 15 second bar, no error is thrown
# Despite the returned direction being -1, this seems to have gone forwards in time
dataQuery(
    conid = 265598,
    bar = '15S',
    period = '1min',
    startTime = datetime(2026, 9, 8, 15, 0, 15),
    outsideRth = False
)

{'serverId': '168821944', 'symbol': 'AAPL', 'text': 'APPLE INC', 'priceFactor': 100, 'startTime': '20260908-14:59:00', 'high': '31603/319.925/3', 'low': '31566/961/0', 'timePeriod': '60s', 'barLength': 15, 'mdAvailability': 'S', 'mktDataDelay': 0, 'outsideRth': False, 'volumeFactor': 40, 'priceDisplayRule': 1, 'priceDisplayValue': '2', 'chartPanStartTime': '20260908-15:00:15', 'direction': -1, 'negativeCapable': False, 'messageVersion': 2, 'data': [{'o': 315.74, 'c': 315.78, 'h': 315.79, 'l': 315.66, 'v': 961, 't': 1788879540000}, {'o': 315.78, 'c': 315.8, 'h': 315.85, 'l': 315.75, 'v': 355.5, 't': 1788879600000}, {'o': 315.78, 'c': 315.89, 'h': 315.91, 'l': 315.74, 'v': 205.4, 't': 1788879660000}, {'o': 315.92, 'c': 315.97, 'h': 316.03, 'l': 315.9, 'v': 319.925, 't': 1788879720000}], 'points': 3, 'travelTime': 403}
startTime:           (utc) 2026-09-08 Tue 02:59:00 PM, (NY) 2026-09-08 Tue 10:59:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 03:00:15 PM, (NY) 2026-09-08 Tue 11:00:15 A

,o,c,h,l,v,t
0,315.74,315.78,315.79,315.66,961.000,"(utc) 2026-09-08 Tue 02:59:00 PM, (NY) 2026-09-08 Tue 10:59:00 AM"
1,315.78,315.80,315.85,315.75,355.500,"(utc) 2026-09-08 Tue 03:00:00 PM, (NY) 2026-09-08 Tue 11:00:00 AM"
2,315.78,315.89,315.91,315.74,205.400,"(utc) 2026-09-08 Tue 03:01:00 PM, (NY) 2026-09-08 Tue 11:01:00 AM"
3,315.92,315.97,316.03,315.90,319.925,"(utc) 2026-09-08 Tue 03:02:00 PM, (NY) 2026-09-08 Tue 11:02:00 AM"


In [ ]:
# See the last query in this section for further analysis
dataQuery(
    conid = 265598,
    bar = '15S',
    period = '5min',
    startTime = datetime(2026, 9, 8, 15, 0, 15),
    outsideRth = False
)

{'serverId': '168771283', 'symbol': 'AAPL', 'text': 'APPLE INC', 'priceFactor': 100, 'startTime': '20260908-14:55:00', 'high': '31638/325.025/5', 'low': '31562/285.4/13', 'timePeriod': '300s', 'barLength': 15, 'mdAvailability': 'S', 'mktDataDelay': 0, 'outsideRth': False, 'volumeFactor': 40, 'priceDisplayRule': 1, 'priceDisplayValue': '2', 'chartPanStartTime': '20260908-15:00:15', 'direction': -1, 'negativeCapable': False, 'messageVersion': 2, 'data': [{'o': 316.29, 'c': 316.31, 'h': 316.33, 'l': 316.29, 'v': 215.55, 't': 1788879300000}, {'o': 316.32, 'c': 316.24, 'h': 316.34, 'l': 316.23, 'v': 224.975, 't': 1788879360000}, {'o': 316.23, 'c': 316.26, 'h': 316.28, 'l': 316.23, 'v': 145.6, 't': 1788879420000}, {'o': 316.26, 'c': 316.33, 'h': 316.35, 'l': 316.23, 'v': 160.1, 't': 1788879480000}, {'o': 316.33, 'c': 316.32, 'h': 316.34, 'l': 316.28, 'v': 186.15, 't': 1788879540000}, {'o': 316.34, 'c': 316.38, 'h': 316.38, 'l': 316.31, 'v': 325.025, 't': 1788879600000}, {'o': 316.36, 'c': 31

,o,c,h,l,v,t
0,316.29,316.31,316.33,316.29,215.550,"(utc) 2026-09-08 Tue 02:55:00 PM, (NY) 2026-09-08 Tue 10:55:00 AM"
1,316.32,316.24,316.34,316.23,224.975,"(utc) 2026-09-08 Tue 02:56:00 PM, (NY) 2026-09-08 Tue 10:56:00 AM"
2,316.23,316.26,316.28,316.23,145.600,"(utc) 2026-09-08 Tue 02:57:00 PM, (NY) 2026-09-08 Tue 10:57:00 AM"
3,316.26,316.33,316.35,316.23,160.100,"(utc) 2026-09-08 Tue 02:58:00 PM, (NY) 2026-09-08 Tue 10:58:00 AM"
4,316.33,316.32,316.34,316.28,186.150,"(utc) 2026-09-08 Tue 02:59:00 PM, (NY) 2026-09-08 Tue 10:59:00 AM"
5,316.34,316.38,316.38,316.31,325.025,"(utc) 2026-09-08 Tue 03:00:00 PM, (NY) 2026-09-08 Tue 11:00:00 AM"
6,316.36,316.29,316.37,316.26,495.925,"(utc) 2026-09-08 Tue 03:01:00 PM, (NY) 2026-09-08 Tue 11:01:00 AM"
7,316.28,316.12,316.30,316.06,427.475,"(utc) 2026-09-08 Tue 03:02:00 PM, (NY) 2026-09-08 Tue 11:02:00 AM"
8,316.11,316.08,316.13,316.07,224.450,"(utc) 2026-09-08 Tue 03:03:00 PM, (NY) 2026-09-08 Tue 11:03:00 AM"
9,316.06,315.93,316.07,315.92,476.750,"(utc) 2026-09-08 Tue 03:04:00 PM, (NY) 2026-09-08 Tue 11:04:00 AM"


In [32]:
# Using an invalid bar size makes the period jump to 10 minutes
dataQuery(
    conid = 265598,
    bar = '22S',
    period = '60S',
    startTime = datetime(2026, 9, 8, 15, 0, 5),
    outsideRth = False
)

{'serverId': '167962973', 'symbol': 'AAPL', 'text': 'APPLE INC', 'priceFactor': 100, 'startTime': '20260908-14:59:00', 'high': '31603/175.15/10', 'low': '31566/141.1/1', 'timePeriod': '60s', 'barLength': 5, 'mdAvailability': 'S', 'mktDataDelay': 0, 'outsideRth': False, 'volumeFactor': 40, 'priceDisplayRule': 1, 'priceDisplayValue': '2', 'chartPanStartTime': '20260908-15:00:05', 'direction': -1, 'negativeCapable': False, 'messageVersion': 2, 'data': [{'o': 315.74, 'c': 315.7, 'h': 315.74, 'l': 315.69, 'v': 13.575, 't': 1788879540000}, {'o': 315.69, 'c': 315.7, 'h': 315.7, 'l': 315.66, 'v': 141.1, 't': 1788879600000}, {'o': 315.69, 'c': 315.78, 'h': 315.79, 'l': 315.69, 'v': 806.325, 't': 1788879660000}, {'o': 315.78, 'c': 315.84, 'h': 315.85, 'l': 315.77, 'v': 190.65, 't': 1788879720000}, {'o': 315.82, 'c': 315.81, 'h': 315.85, 'l': 315.8, 'v': 56.375, 't': 1788879780000}, {'o': 315.8, 'c': 315.8, 'h': 315.81, 'l': 315.75, 'v': 108.475, 't': 1788879840000}, {'o': 315.78, 'c': 315.76, 'h

,o,c,h,l,v,t
0,315.74,315.70,315.74,315.69,13.575,"(utc) 2026-09-08 Tue 02:59:00 PM, (NY) 2026-09-08 Tue 10:59:00 AM"
1,315.69,315.70,315.70,315.66,141.100,"(utc) 2026-09-08 Tue 03:00:00 PM, (NY) 2026-09-08 Tue 11:00:00 AM"
2,315.69,315.78,315.79,315.69,806.325,"(utc) 2026-09-08 Tue 03:01:00 PM, (NY) 2026-09-08 Tue 11:01:00 AM"
3,315.78,315.84,315.85,315.77,190.650,"(utc) 2026-09-08 Tue 03:02:00 PM, (NY) 2026-09-08 Tue 11:02:00 AM"
4,315.82,315.81,315.85,315.80,56.375,"(utc) 2026-09-08 Tue 03:03:00 PM, (NY) 2026-09-08 Tue 11:03:00 AM"
5,315.80,315.80,315.81,315.75,108.475,"(utc) 2026-09-08 Tue 03:04:00 PM, (NY) 2026-09-08 Tue 11:04:00 AM"
6,315.78,315.76,315.80,315.75,58.275,"(utc) 2026-09-08 Tue 03:05:00 PM, (NY) 2026-09-08 Tue 11:05:00 AM"
7,315.76,315.76,315.79,315.74,58.900,"(utc) 2026-09-08 Tue 03:06:00 PM, (NY) 2026-09-08 Tue 11:06:00 AM"
8,315.80,315.89,315.91,315.80,88.225,"(utc) 2026-09-08 Tue 03:07:00 PM, (NY) 2026-09-08 Tue 11:07:00 AM"
9,315.92,315.98,315.99,315.90,66.550,"(utc) 2026-09-08 Tue 03:08:00 PM, (NY) 2026-09-08 Tue 11:08:00 AM"


In [ ]:
# Now with a normal 1 min query to compare with, the only value that aligns is the 10:59 open at 315.74
# Though notably if we compare the close of the first query with 15s bars we see it corresponds to the close of the 10:59 bar, which implies that potentially the timings returned are wrong and that the 15s bar is actually 15s. 
# This theory is futher backed up by the second query where we see bar 8->11, 12->15 and 16->19 correspond with the results of this query
# Some bars such as 11:01 have very big differences between the 3 queries (if just comparing directly using the time returned)
dataQuery(
    conid = 265598,
    bar = '1min',
    period = '5min',
    startTime = datetime(2026, 9, 8, 15, 3),
    outsideRth = False
)

{'serverId': '167833591', 'symbol': 'AAPL', 'text': 'APPLE INC', 'priceFactor': 100, 'startTime': '20260908-14:57:00', 'high': '31615/1121.925/3', 'low': '31562/1509.2/1', 'timePeriod': '300s', 'barLength': 60, 'mdAvailability': 'S', 'mktDataDelay': 0, 'outsideRth': False, 'volumeFactor': 40, 'priceDisplayRule': 1, 'priceDisplayValue': '2', 'chartPanStartTime': '20260908-15:03:00', 'direction': -1, 'negativeCapable': False, 'messageVersion': 2, 'data': [{'o': 316.11, 'c': 315.76, 'h': 316.13, 'l': 315.72, 'v': 1479.175, 't': 1788879420000}, {'o': 315.76, 'c': 315.72, 'h': 315.82, 'l': 315.62, 'v': 1509.2, 't': 1788879480000}, {'o': 315.74, 'c': 315.97, 'h': 316.03, 'l': 315.66, 'v': 1841.825, 't': 1788879540000}, {'o': 316.0, 'c': 315.99, 'h': 316.15, 'l': 315.93, 'v': 1121.925, 't': 1788879600000}, {'o': 316.03, 'c': 316.03, 'h': 316.13, 'l': 315.93, 'v': 796.475, 't': 1788879660000}], 'points': 4, 'travelTime': 204}
startTime:           (utc) 2026-09-08 Tue 02:57:00 PM, (NY) 2026-09-

,o,c,h,l,v,t
0,316.11,315.76,316.13,315.72,1479.175,"(utc) 2026-09-08 Tue 02:57:00 PM, (NY) 2026-09-08 Tue 10:57:00 AM"
1,315.76,315.72,315.82,315.62,1509.200,"(utc) 2026-09-08 Tue 02:58:00 PM, (NY) 2026-09-08 Tue 10:58:00 AM"
2,315.74,315.97,316.03,315.66,1841.825,"(utc) 2026-09-08 Tue 02:59:00 PM, (NY) 2026-09-08 Tue 10:59:00 AM"
3,316.00,315.99,316.15,315.93,1121.925,"(utc) 2026-09-08 Tue 03:00:00 PM, (NY) 2026-09-08 Tue 11:00:00 AM"
4,316.03,316.03,316.13,315.93,796.475,"(utc) 2026-09-08 Tue 03:01:00 PM, (NY) 2026-09-08 Tue 11:01:00 AM"


Fetching too many bars

In [3]:
# This just succeeds but trims the output
dataQuery(
    conid = 265598,
    bar = '1min',
    period = '1m',
    startTime = datetime(2026, 9, 8, 15, 3),
    outsideRth = False
)

startTime:           (utc) 2026-09-02 Wed 05:52:00 PM, (NY) 2026-09-02 Wed 01:52:00 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 03:03:00 PM, (NY) 2026-09-08 Tue 11:03:00 AM
timePeriod:          1m
barLength:           60


,o,c,h,l,v,t
0,325.50,325.59,325.60,325.45,426.350,"(utc) 2026-09-02 Wed 05:52:00 PM, (NY) 2026-09-02 Wed 01:52:00 PM"
1,325.60,325.61,325.64,325.56,464.475,"(utc) 2026-09-02 Wed 05:53:00 PM, (NY) 2026-09-02 Wed 01:53:00 PM"
2,325.61,325.83,325.98,325.61,1381.350,"(utc) 2026-09-02 Wed 05:54:00 PM, (NY) 2026-09-02 Wed 01:54:00 PM"
3,325.83,325.91,325.98,325.76,881.675,"(utc) 2026-09-02 Wed 05:55:00 PM, (NY) 2026-09-02 Wed 01:55:00 PM"
4,325.91,325.99,326.01,325.86,641.500,"(utc) 2026-09-02 Wed 05:56:00 PM, (NY) 2026-09-02 Wed 01:56:00 PM"
...,...,...,...,...,...,...
995,316.11,315.76,316.13,315.72,1479.175,"(utc) 2026-09-08 Tue 02:57:00 PM, (NY) 2026-09-08 Tue 10:57:00 AM"
996,315.76,315.72,315.82,315.62,1509.200,"(utc) 2026-09-08 Tue 02:58:00 PM, (NY) 2026-09-08 Tue 10:58:00 AM"
997,315.74,315.97,316.03,315.66,1841.825,"(utc) 2026-09-08 Tue 02:59:00 PM, (NY) 2026-09-08 Tue 10:59:00 AM"
998,316.00,315.99,316.15,315.93,1121.925,"(utc) 2026-09-08 Tue 03:00:00 PM, (NY) 2026-09-08 Tue 11:00:00 AM"


In [3]:
# This one fails as after 10 seconds the query is cancelled
dataQuery(
    conid = 265598,
    bar = '1min',
    period = '1y',
    startTime = datetime(2026, 9, 8, 15, 3),
    outsideRth = False
)

KeyError: 'startTime'

Different Time Zones returned by API

In [ ]:
# Start of bar is the exact start time of trading day in local timezone
dataQuery(
    conid = 265598,
    bar = '1d',
    period = '1w',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
)

startTime:           (utc) 2026-04-23 Thu 01:30:00 PM, (NY) 2026-04-23 Thu 09:30:00 AM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          1w
barLength:           86400


,o,c,h,l,v,t
0,272.79,271.06,273.06,269.65,457177.875,"(utc) 2026-04-24 Fri 01:30:00 PM, (NY) 2026-04-24 Fri 09:30:00 AM"
1,266.09,267.61,268.36,265.07,475229.000,"(utc) 2026-04-27 Mon 01:30:00 PM, (NY) 2026-04-27 Mon 09:30:00 AM"
2,272.34,270.71,273.20,268.66,438889.675,"(utc) 2026-04-28 Tue 01:30:00 PM, (NY) 2026-04-28 Tue 09:30:00 AM"
3,267.59,270.17,271.04,267.04,333112.675,"(utc) 2026-04-29 Wed 01:30:00 PM, (NY) 2026-04-29 Wed 09:30:00 AM"


In [ ]:
# However, with larger bars this switches to being at 12am UTC of the given period
# Note: it won't always be the first day of period if there is no trading on that date
dataQuery(
    conid = 265598,
    bar = '1w',
    period = '1m',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
)

startTime:           (utc) 2026-03-24 Tue 12:00:00 AM, (NY) 2026-03-23 Mon 08:00:00 PM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          1m
barLength:           604800


,o,c,h,l,v,t
0,250.35,248.80,257.00,248.07,1853233.000,"(utc) 2026-03-24 Tue 12:00:00 AM, (NY) 2026-03-23 Mon 08:00:00 PM"
1,250.05,255.92,256.17,245.51,1752755.750,"(utc) 2026-03-30 Mon 12:00:00 AM, (NY) 2026-03-29 Sun 08:00:00 PM"
2,256.51,260.48,262.19,245.70,2517735.850,"(utc) 2026-04-06 Mon 12:00:00 AM, (NY) 2026-04-05 Sun 08:00:00 PM"
3,259.60,270.23,272.30,256.66,2937499.500,"(utc) 2026-04-13 Mon 12:00:00 AM, (NY) 2026-04-12 Sun 08:00:00 PM"
4,270.33,273.17,274.28,265.40,1779958.025,"(utc) 2026-04-20 Mon 12:00:00 AM, (NY) 2026-04-19 Sun 08:00:00 PM"


Different periods, same number of bars returned

In [49]:
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '16m',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
)

startTime:           (utc) 2024-12-16 Mon 12:00:00 AM, (NY) 2024-12-15 Sun 07:00:00 PM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          16m
barLength:           2678400


,o,c,h,l,v,t
0,247.99,250.42,260.03,245.69,6.107112e+06,"(utc) 2024-12-16 Mon 12:00:00 AM, (NY) 2024-12-15 Sun 07:00:00 PM"
1,248.93,236.00,249.10,219.38,1.669847e+07,"(utc) 2025-01-02 Thu 12:00:00 AM, (NY) 2025-01-01 Wed 07:00:00 PM"
2,230.00,241.84,250.00,225.70,1.156510e+07,"(utc) 2025-02-03 Mon 12:00:00 AM, (NY) 2025-02-02 Sun 07:00:00 PM"
3,241.81,222.13,244.03,208.42,1.554193e+07,"(utc) 2025-03-03 Mon 12:00:00 AM, (NY) 2025-03-02 Sun 07:00:00 PM"
4,219.76,212.50,225.19,169.21,2.330168e+07,"(utc) 2025-04-01 Tue 12:00:00 AM, (NY) 2025-03-31 Mon 08:00:00 PM"
5,208.90,200.85,214.56,193.25,1.650167e+07,"(utc) 2025-05-01 Thu 12:00:00 AM, (NY) 2025-04-30 Wed 08:00:00 PM"
6,200.28,205.17,207.39,195.07,1.557226e+07,"(utc) 2025-06-02 Mon 12:00:00 AM, (NY) 2025-06-01 Sun 08:00:00 PM"
7,206.72,207.57,216.23,206.14,1.578090e+07,"(utc) 2025-07-01 Tue 12:00:00 AM, (NY) 2025-06-30 Mon 08:00:00 PM"
8,210.95,232.14,235.06,201.50,1.789998e+07,"(utc) 2025-08-01 Fri 12:00:00 AM, (NY) 2025-07-31 Thu 08:00:00 PM"
9,229.25,254.63,257.60,225.95,1.783923e+07,"(utc) 2025-09-02 Tue 12:00:00 AM, (NY) 2025-09-01 Mon 08:00:00 PM"


In [50]:
# Despite adding another month of data to our period, same number of bars are returned. Start dates are different
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '17m',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
)

startTime:           (utc) 2024-11-18 Mon 12:00:00 AM, (NY) 2024-11-17 Sun 07:00:00 PM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          17m
barLength:           2678400


,o,c,h,l,v,t
0,237.27,250.42,260.03,237.16,1.109321e+07,"(utc) 2024-12-02 Mon 12:00:00 AM, (NY) 2024-12-01 Sun 07:00:00 PM"
1,248.93,236.00,249.10,219.38,1.669847e+07,"(utc) 2025-01-02 Thu 12:00:00 AM, (NY) 2025-01-01 Wed 07:00:00 PM"
2,230.00,241.84,250.00,225.70,1.156510e+07,"(utc) 2025-02-03 Mon 12:00:00 AM, (NY) 2025-02-02 Sun 07:00:00 PM"
3,241.81,222.13,244.03,208.42,1.554193e+07,"(utc) 2025-03-03 Mon 12:00:00 AM, (NY) 2025-03-02 Sun 07:00:00 PM"
4,219.76,212.50,225.19,169.21,2.330168e+07,"(utc) 2025-04-01 Tue 12:00:00 AM, (NY) 2025-03-31 Mon 08:00:00 PM"
5,208.90,200.85,214.56,193.25,1.650167e+07,"(utc) 2025-05-01 Thu 12:00:00 AM, (NY) 2025-04-30 Wed 08:00:00 PM"
6,200.28,205.17,207.39,195.07,1.557226e+07,"(utc) 2025-06-02 Mon 12:00:00 AM, (NY) 2025-06-01 Sun 08:00:00 PM"
7,206.72,207.57,216.23,206.14,1.578090e+07,"(utc) 2025-07-01 Tue 12:00:00 AM, (NY) 2025-06-30 Mon 08:00:00 PM"
8,210.95,232.14,235.06,201.50,1.789998e+07,"(utc) 2025-08-01 Fri 12:00:00 AM, (NY) 2025-07-31 Thu 08:00:00 PM"
9,229.25,254.63,257.60,225.95,1.783923e+07,"(utc) 2025-09-02 Tue 12:00:00 AM, (NY) 2025-09-01 Mon 08:00:00 PM"


In [4]:
# Even more weridly, a one month bar over a period of one month returns 2 bars
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1m',
    startTime = datetime(2026, 3, 31, 15, 0),
    outsideRth = False
)

startTime:           (utc) 2026-02-09 Mon 12:00:00 AM, (NY) 2026-02-08 Sun 07:00:00 PM
chartPanStartTime:   (utc) 2026-03-31 Tue 03:00:00 PM, (NY) 2026-03-31 Tue 11:00:00 AM
timePeriod:          1m
barLength:           2678400


,o,c,h,l,v,t
0,277.87,264.18,280.18,255.45,7882300.350,"(utc) 2026-02-09 Mon 12:00:00 AM, (NY) 2026-02-08 Sun 07:00:00 PM"
1,262.46,259.88,266.53,253.68,2870816.875,"(utc) 2026-03-02 Mon 12:00:00 AM, (NY) 2026-03-01 Sun 07:00:00 PM"


Partial Bars are inconsistent, the last bar returned is usually full however the earliest one is often partial. We also cannot get a full 1m bar starting on any day of our choosing (we can't have 1m bars starting from say the 10th of every month)

In [5]:
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1m',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
)

startTime:           (utc) 2026-03-11 Wed 12:00:00 AM, (NY) 2026-03-10 Tue 08:00:00 PM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          1m
barLength:           2678400


,o,c,h,l,v,t
0,261.11,253.79,262.13,245.51,6706945.975,"(utc) 2026-03-11 Wed 12:00:00 AM, (NY) 2026-03-10 Tue 08:00:00 PM"
1,253.90,260.49,262.16,245.70,2942747.425,"(utc) 2026-04-01 Wed 12:00:00 AM, (NY) 2026-03-31 Tue 08:00:00 PM"


The latest bar returned for large bars can also be a partial bar

In [5]:
# These are the earliest results from a query (we know these can have partial bars as is the case with 1991-06-20)
dataQuery(
    conid = 265598,
    bar = '1w',
    period = '210m',
    startTime = datetime(2008, 9, 22, 0, 0),
    outsideRth = False
).head(3)

startTime:           (utc) 1991-06-20 Thu 12:00:00 AM, (NY) 1991-06-19 Wed 08:00:00 PM
chartPanStartTime:   (utc) 2008-09-22 Mon 12:00:00 AM, (NY) 2008-09-21 Sun 08:00:00 PM
timePeriod:          210m
barLength:           604800


,o,c,h,l,v,t
0,1.50,1.50,1.52,1.46,2187850,"(utc) 1991-06-20 Thu 12:00:00 AM, (NY) 1991-06-19 Wed 08:00:00 PM"
1,1.49,1.48,1.55,1.44,6651190,"(utc) 1991-06-24 Mon 12:00:00 AM, (NY) 1991-06-23 Sun 08:00:00 PM"
2,1.52,1.63,1.64,1.49,5945590,"(utc) 1991-07-01 Mon 12:00:00 AM, (NY) 1991-06-30 Sun 08:00:00 PM"


In [6]:
# However in this follow up query the latest 1991-06-24 bar doesn't capture all the information it should (compared to the "full" bar above)
dataQuery(
    conid = 265598,
    bar = '1w',
    period = '210m',
    startTime = datetime(1991, 7, 1, 0, 0),
    outsideRth = False
).tail(3)

startTime:           (utc) 1980-12-12 Fri 12:00:00 AM, (NY) 1980-12-11 Thu 07:00:00 PM
chartPanStartTime:   (utc) 1991-07-01 Mon 12:00:00 AM, (NY) 1991-06-30 Sun 08:00:00 PM
timePeriod:          210m
barLength:           604800


,o,c,h,l,v,t
547,1.64,1.47,1.68,1.46,7676830,"(utc) 1991-06-10 Mon 12:00:00 AM, (NY) 1991-06-09 Sun 08:00:00 PM"
548,1.50,1.50,1.54,1.46,5876780,"(utc) 1991-06-17 Mon 12:00:00 AM, (NY) 1991-06-16 Sun 08:00:00 PM"
549,1.49,1.54,1.55,1.47,4289670,"(utc) 1991-06-24 Mon 12:00:00 AM, (NY) 1991-06-23 Sun 08:00:00 PM"


Year period with days seems to match calendar year (potentially ≈251 trading days)

In [4]:
dataQuery(
    conid = 265598,
    bar = '1d',
    period = '1y',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
).head()

startTime:           (utc) 2025-04-30 Wed 01:30:00 PM, (NY) 2025-04-30 Wed 09:30:00 AM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          1y
barLength:           86400


,o,c,h,l,v,t
0,208.90,213.32,214.56,208.90,669859.975,"(utc) 2025-05-01 Thu 01:30:00 PM, (NY) 2025-05-01 Thu 09:30:00 AM"
1,206.09,205.35,206.99,202.16,1474746.250,"(utc) 2025-05-02 Fri 01:30:00 PM, (NY) 2025-05-02 Fri 09:30:00 AM"
2,203.10,198.89,204.10,198.21,951988.925,"(utc) 2025-05-05 Mon 01:30:00 PM, (NY) 2025-05-05 Mon 09:30:00 AM"
3,198.17,198.51,200.65,197.02,669145.725,"(utc) 2025-05-06 Tue 01:30:00 PM, (NY) 2025-05-06 Tue 09:30:00 AM"
4,199.11,196.25,199.44,193.25,895162.700,"(utc) 2025-05-07 Wed 01:30:00 PM, (NY) 2025-05-07 Wed 09:30:00 AM"


In [8]:
dataQuery(
    conid = 265598,
    bar = '1d',
    period = '1y',
    startTime = datetime(2026, 4, 28, 6, 0),
    outsideRth = False
).head()

startTime:           (utc) 2025-04-25 Fri 01:30:00 PM, (NY) 2025-04-25 Fri 09:30:00 AM
chartPanStartTime:   (utc) 2026-04-28 Tue 06:00:00 AM, (NY) 2026-04-28 Tue 02:00:00 AM
timePeriod:          1y
barLength:           86400


,o,c,h,l,v,t
0,210.06,210.14,211.50,207.46,475926.250,"(utc) 2025-04-28 Mon 01:30:00 PM, (NY) 2025-04-28 Mon 09:30:00 AM"
1,208.80,211.21,212.24,208.37,416177.850,"(utc) 2025-04-29 Tue 01:30:00 PM, (NY) 2025-04-29 Tue 09:30:00 AM"
2,209.26,212.50,213.58,206.67,521586.025,"(utc) 2025-04-30 Wed 01:30:00 PM, (NY) 2025-04-30 Wed 09:30:00 AM"
3,208.90,213.32,214.56,208.90,669859.975,"(utc) 2025-05-01 Thu 01:30:00 PM, (NY) 2025-05-01 Thu 09:30:00 AM"
4,206.09,205.35,206.99,202.16,1474746.250,"(utc) 2025-05-02 Fri 01:30:00 PM, (NY) 2025-05-02 Fri 09:30:00 AM"


In [ ]:
# Trying in a completely different year
dataQuery(
    conid = 265598,
    bar = '1d',
    period = '1y',
    startTime = datetime(2022, 4, 28, 6, 0),
    outsideRth = False
).head()

startTime:           (utc) 2021-04-27 Tue 01:30:00 PM, (NY) 2021-04-27 Tue 09:30:00 AM
chartPanStartTime:   (utc) 2022-04-28 Thu 06:00:00 AM, (NY) 2022-04-28 Thu 02:00:00 AM
timePeriod:          1y
barLength:           86400


,o,c,h,l,v,t
0,134.31,133.58,135.01,133.08,1784716.975,"(utc) 2021-04-28 Wed 01:30:00 PM, (NY) 2021-04-28 Wed 09:30:00 AM"
1,136.46,133.48,137.07,132.45,3011036.075,"(utc) 2021-04-29 Thu 01:30:00 PM, (NY) 2021-04-29 Thu 09:30:00 AM"
2,131.80,131.46,133.56,131.06,1877656.825,"(utc) 2021-04-30 Fri 01:30:00 PM, (NY) 2021-04-30 Fri 09:30:00 AM"
3,132.01,132.54,134.07,131.83,1496184.150,"(utc) 2021-05-03 Mon 01:30:00 PM, (NY) 2021-05-03 Mon 09:30:00 AM"
4,131.18,127.85,131.49,126.70,2632548.525,"(utc) 2021-05-04 Tue 01:30:00 PM, (NY) 2021-05-04 Tue 09:30:00 AM"


In [21]:
# Seems to be consistent even over longer periods
dataQuery(
    conid = 265598,
    bar = '1d',
    period = '2y',
    startTime = datetime(2022, 8, 11, 6, 0),
    outsideRth = False
).head()

startTime:           (utc) 2020-08-10 Mon 01:30:00 PM, (NY) 2020-08-10 Mon 09:30:00 AM
chartPanStartTime:   (utc) 2022-08-11 Thu 06:00:00 AM, (NY) 2022-08-11 Thu 02:00:00 AM
timePeriod:          2y
barLength:           86400


,o,c,h,l,v,t
0,111.92,109.38,112.48,109.10,2955983.0,"(utc) 2020-08-11 Tue 01:30:00 PM, (NY) 2020-08-11 Tue 09:30:00 AM"
1,110.44,113.01,113.28,110.30,3043575.3,"(utc) 2020-08-12 Wed 01:30:00 PM, (NY) 2020-08-12 Wed 09:30:00 AM"
2,114.44,115.01,116.04,113.93,3875172.7,"(utc) 2020-08-13 Thu 01:30:00 PM, (NY) 2020-08-13 Thu 09:30:00 AM"
3,114.74,114.91,115.00,113.04,3031509.7,"(utc) 2020-08-14 Fri 01:30:00 PM, (NY) 2020-08-14 Fri 09:30:00 AM"
4,116.04,114.61,116.09,113.96,1980974.7,"(utc) 2020-08-17 Mon 01:30:00 PM, (NY) 2020-08-17 Mon 09:30:00 AM"


However, year period with months displays arbitrary "snapping" behaviour

In [40]:
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1y',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
).head(3)

startTime:           (utc) 2025-04-10 Thu 12:00:00 AM, (NY) 2025-04-09 Wed 08:00:00 PM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          1y
barLength:           2678400


,o,c,h,l,v,t
0,189.06,212.50,213.58,183.00,1.162514e+07,"(utc) 2025-04-10 Thu 12:00:00 AM, (NY) 2025-04-09 Wed 08:00:00 PM"
1,208.90,200.85,214.56,193.25,1.650167e+07,"(utc) 2025-05-01 Thu 12:00:00 AM, (NY) 2025-04-30 Wed 08:00:00 PM"
2,200.28,205.17,207.39,195.07,1.557226e+07,"(utc) 2025-06-02 Mon 12:00:00 AM, (NY) 2025-06-01 Sun 08:00:00 PM"


In [41]:
# 5 days earlier still snaps to same date
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1y',
    startTime = datetime(2026, 4, 25, 15, 0),
    outsideRth = False
).head(3)

startTime:           (utc) 2025-04-10 Thu 12:00:00 AM, (NY) 2025-04-09 Wed 08:00:00 PM
chartPanStartTime:   (utc) 2026-04-25 Sat 03:00:00 PM, (NY) 2026-04-25 Sat 11:00:00 AM
timePeriod:          1y
barLength:           2678400


,o,c,h,l,v,t
0,189.06,212.50,213.58,183.00,1.162514e+07,"(utc) 2025-04-10 Thu 12:00:00 AM, (NY) 2025-04-09 Wed 08:00:00 PM"
1,208.90,200.85,214.56,193.25,1.650167e+07,"(utc) 2025-05-01 Thu 12:00:00 AM, (NY) 2025-04-30 Wed 08:00:00 PM"
2,200.28,205.17,207.39,195.07,1.557226e+07,"(utc) 2025-06-02 Mon 12:00:00 AM, (NY) 2025-06-01 Sun 08:00:00 PM"


In [42]:
# However 6 days earlier snaps to a different date and returns a different partial bar
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1y',
    startTime = datetime(2026, 4, 24, 15, 0),
    outsideRth = False
).head(3)

startTime:           (utc) 2025-03-10 Mon 12:00:00 AM, (NY) 2025-03-09 Sun 08:00:00 PM
chartPanStartTime:   (utc) 2026-04-24 Fri 03:00:00 PM, (NY) 2026-04-24 Fri 11:00:00 AM
timePeriod:          1y
barLength:           2678400


,o,c,h,l,v,t
0,235.50,222.13,236.16,208.42,1.207336e+07,"(utc) 2025-03-10 Mon 12:00:00 AM, (NY) 2025-03-09 Sun 08:00:00 PM"
1,219.76,212.50,225.19,169.21,2.330168e+07,"(utc) 2025-04-01 Tue 12:00:00 AM, (NY) 2025-03-31 Mon 08:00:00 PM"
2,208.90,200.85,214.56,193.25,1.650167e+07,"(utc) 2025-05-01 Thu 12:00:00 AM, (NY) 2025-04-30 Wed 08:00:00 PM"


In [43]:
# No, it doesn't always snap to the 10th of the month
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1y',
    startTime = datetime(2025, 8, 11, 15, 0),
    outsideRth = False
).head(3)

startTime:           (utc) 2024-07-05 Fri 12:00:00 AM, (NY) 2024-07-04 Thu 08:00:00 PM
chartPanStartTime:   (utc) 2025-08-11 Mon 03:00:00 PM, (NY) 2025-08-11 Mon 11:00:00 AM
timePeriod:          1y
barLength:           2678400


,o,c,h,l,v,t
0,221.65,222.08,237.23,214.62,1.556370e+07,"(utc) 2024-07-05 Fri 12:00:00 AM, (NY) 2024-07-04 Thu 08:00:00 PM"
1,224.37,229.00,232.92,196.21,1.579085e+07,"(utc) 2024-08-01 Thu 12:00:00 AM, (NY) 2024-07-31 Wed 08:00:00 PM"
2,228.62,233.00,233.09,213.92,1.478058e+07,"(utc) 2024-09-03 Tue 12:00:00 AM, (NY) 2024-09-02 Mon 08:00:00 PM"


In [44]:
# The partial bars behaviour is inconsistent, sometimes it locks onto a correct day
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '5y',
    startTime = datetime(2025, 8, 11, 15, 0),
    outsideRth = False
).head(3)

startTime:           (utc) 2020-08-31 Mon 12:00:00 AM, (NY) 2020-08-30 Sun 08:00:00 PM
chartPanStartTime:   (utc) 2025-08-11 Mon 03:00:00 PM, (NY) 2025-08-11 Mon 11:00:00 AM
timePeriod:          5y
barLength:           2678400


,o,c,h,l,v,t
0,132.79,115.81,137.97,103.10,7.535771e+07,"(utc) 2020-09-01 Tue 12:00:00 AM, (NY) 2020-08-31 Mon 08:00:00 PM"
1,117.70,108.86,125.39,107.72,5.838150e+07,"(utc) 2020-10-01 Thu 12:00:00 AM, (NY) 2020-09-30 Wed 08:00:00 PM"
2,109.14,119.05,121.99,107.32,4.116370e+07,"(utc) 2020-11-02 Mon 12:00:00 AM, (NY) 2020-11-01 Sun 07:00:00 PM"


In [45]:
# But other times it does not
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '5y',
    startTime = datetime(2025, 9, 21, 15, 0),
    outsideRth = False
).head(3)

startTime:           (utc) 2020-09-08 Tue 12:00:00 AM, (NY) 2020-09-07 Mon 08:00:00 PM
chartPanStartTime:   (utc) 2025-09-21 Sun 03:00:00 PM, (NY) 2025-09-21 Sun 11:00:00 AM
timePeriod:          5y
barLength:           2678400


,o,c,h,l,v,t
0,114.28,115.81,120.50,103.10,5.741944e+07,"(utc) 2020-09-08 Tue 12:00:00 AM, (NY) 2020-09-07 Mon 08:00:00 PM"
1,117.70,108.86,125.39,107.72,5.838150e+07,"(utc) 2020-10-01 Thu 12:00:00 AM, (NY) 2020-09-30 Wed 08:00:00 PM"
2,109.14,119.05,121.99,107.32,4.116370e+07,"(utc) 2020-11-02 Mon 12:00:00 AM, (NY) 2020-11-01 Sun 07:00:00 PM"


The latest bar returned also experiences snapping, but returns a full set of data regardless of the date entered  
The API doesn't guarantee that the latest bar returned also covers the date entered to the api

In [65]:
# February not in returned datea
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1y',
    startTime = datetime(2026, 2, 21, 15, 0),
    outsideRth = False
).tail(5)

startTime:           (utc) 2025-01-07 Tue 12:00:00 AM, (NY) 2025-01-06 Mon 07:00:00 PM
chartPanStartTime:   (utc) 2026-02-21 Sat 03:00:00 PM, (NY) 2026-02-21 Sat 10:00:00 AM
timePeriod:          1y
barLength:           2678400


,o,c,h,l,v,t
8,229.25,254.63,257.60,225.95,1.783923e+07,"(utc) 2025-09-02 Tue 12:00:00 AM, (NY) 2025-09-01 Mon 08:00:00 PM"
9,255.04,270.37,277.32,244.65,1.488729e+07,"(utc) 2025-10-01 Wed 12:00:00 AM, (NY) 2025-09-30 Tue 08:00:00 PM"
10,270.48,278.85,280.38,265.32,1.072086e+07,"(utc) 2025-11-03 Mon 12:00:00 AM, (NY) 2025-11-02 Sun 07:00:00 PM"
11,278.10,271.86,288.62,266.95,1.050497e+07,"(utc) 2025-12-01 Mon 12:00:00 AM, (NY) 2025-11-30 Sun 07:00:00 PM"
12,272.25,262.36,277.84,262.12,1.817047e+06,"(utc) 2026-01-02 Fri 12:00:00 AM, (NY) 2026-01-01 Thu 07:00:00 PM"


In [64]:
# February now appears in returned data
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '1y',
    startTime = datetime(2026, 2, 22, 15, 0),
    outsideRth = False
).tail(5)

startTime:           (utc) 2025-02-07 Fri 12:00:00 AM, (NY) 2025-02-06 Thu 07:00:00 PM
chartPanStartTime:   (utc) 2026-02-22 Sun 03:00:00 PM, (NY) 2026-02-22 Sun 10:00:00 AM
timePeriod:          1y
barLength:           2678400


,o,c,h,l,v,t
8,255.04,270.37,277.32,244.65,1.488729e+07,"(utc) 2025-10-01 Wed 12:00:00 AM, (NY) 2025-09-30 Tue 08:00:00 PM"
9,270.48,278.85,280.38,265.32,1.072086e+07,"(utc) 2025-11-03 Mon 12:00:00 AM, (NY) 2025-11-02 Sun 07:00:00 PM"
10,278.10,271.86,288.62,266.95,1.050497e+07,"(utc) 2025-12-01 Mon 12:00:00 AM, (NY) 2025-11-30 Sun 07:00:00 PM"
11,272.25,259.48,277.84,243.42,1.320779e+07,"(utc) 2026-01-02 Fri 12:00:00 AM, (NY) 2026-01-01 Thu 07:00:00 PM"
12,260.02,278.12,280.91,259.20,4.478738e+06,"(utc) 2026-02-02 Mon 12:00:00 AM, (NY) 2026-02-01 Sun 07:00:00 PM"


In [12]:
# This applies to smaller bars too, we would expect the last bar to start at 2:59
dataQuery(
    conid = 265598,
    bar = '1min',
    period = '15min',
    startTime = datetime(2026, 4, 30, 15, 0),
    outsideRth = False
).tail(3)

startTime:           (utc) 2026-04-30 Thu 02:44:00 PM, (NY) 2026-04-30 Thu 10:44:00 AM
chartPanStartTime:   (utc) 2026-04-30 Thu 03:00:00 PM, (NY) 2026-04-30 Thu 11:00:00 AM
timePeriod:          900s
barLength:           60


,o,c,h,l,v,t
12,271.25,271.24,271.26,271.16,911.850,"(utc) 2026-04-30 Thu 02:56:00 PM, (NY) 2026-04-30 Thu 10:56:00 AM"
13,271.25,271.21,271.28,271.14,717.600,"(utc) 2026-04-30 Thu 02:57:00 PM, (NY) 2026-04-30 Thu 10:57:00 AM"
14,271.21,271.17,271.24,271.15,797.275,"(utc) 2026-04-30 Thu 02:58:00 PM, (NY) 2026-04-30 Thu 10:58:00 AM"
